### passive sentences

In [5]:
import pandas as pd

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_IN_v1_1.csv"
file2 = "sentence_builders.csv"

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1)
builder = pd.read_csv(file2)

items.columns = items.columns.str.strip()
builder.columns = builder.columns.str.strip()

# Clean FILE1
for col in items.columns:
    if items[col].dtype == "object":
        items[col] = items[col].astype(str).str.strip()

# Clean FILE2
for col in builder.columns:
    if builder[col].dtype == "object":
        builder[col] = builder[col].astype(str).str.strip()

time_specs = (
    builder["time_specs"]
    .replace("nan", pd.NA)
    .dropna()
)
time_specs = time_specs[time_specs != ""].unique().tolist()

adverbs = (
    builder["adverbs"]
    .replace("nan", pd.NA)
    .dropna()
)
adverbs = adverbs[adverbs != ""].unique().tolist()

# =========================
# PAST PARTICIPLES
# =========================

past_participles = {
    "have": "had",
    "receive": "received",
    "take": "taken",
    "make": "made",
    "give": "given"
}

def get_past_participle(verb):
    verb = verb.strip().lower()
    return past_participles.get(verb, verb + "ed")

# =========================
# OBJECT PHRASE
# =========================

def build_object_phrase(row):
    obj = row["object"].strip()
    det = str(row["has_determiner"]).strip().lower()

    if det == "no":
        return obj
    else:
        return f"{det} {obj}"

# =========================
# AGREEMENT
# =========================

def get_past_aux(obj_number):
    obj_number = str(obj_number).strip().lower()

    if obj_number == "pl":
        return "were"
    else:
        return "was"

# =========================
# BUILD FILES
# =========================

output_rows = {
    "past_simple_no_adverbs": [],
    "past_simple_with_adverbs": [],
    "future_simple_no_adverbs": [],
    "future_simple_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

for _, item in items.iterrows():

    verb = item["verb"].strip().lower()
    obj = build_object_phrase(item)
    obj_number = item["obj_number"]

    verb_pp = get_past_participle(verb)
    past_aux = get_past_aux(obj_number)

    for time_spec in time_specs:

        tense_patterns = {
            "past_simple": f"{obj} {past_aux} {{adverb_part}}{verb_pp}",
            "future_simple": f"{obj} will be {{adverb_part}}{verb_pp}",
            "past_perfect": f"{obj} had been {{adverb_part}}{verb_pp}",
        }

        # -------------------------
        # WITHOUT ADVERBS
        # -------------------------
        for tense, pattern in tense_patterns.items():

            sentence_body = pattern.format(adverb_part="")
            sentence = f"{time_spec} {sentence_body}."

            row = item.to_dict()
            row.update({
                "sentence": sentence,
                "voice": "passive",
                "tense": tense,
                "adverb": "NA",
                "time_spec": time_spec
            })

            output_rows[f"{tense}_no_adverbs"].append(row)

        # -------------------------
        # WITH ADVERBS
        # -------------------------
        for adverb in adverbs:
            for tense, pattern in tense_patterns.items():

                sentence_body = pattern.format(adverb_part=f"{adverb} ")
                sentence = f"{time_spec} {sentence_body}."

                row = item.to_dict()
                row.update({
                    "sentence": sentence,
                    "voice": "passive",
                    "tense": tense,
                    "adverb": adverb,
                    "time_spec": time_spec
                })

                output_rows[f"{tense}_with_adverbs"].append(row)

# =========================
# SAVE FILES
# =========================

for name, rows in output_rows.items():

    out = pd.DataFrame(rows)

    # Remove fully empty columns
    out = out.dropna(axis=1, how="all")

    # Remove columns containing only empty strings
    empty_cols = []

    for col in out.columns:

        values = (
            out[col]
            .astype(str)
            .str.strip()
            .replace("nan", "")
        )

        if (values == "").all():
            empty_cols.append(col)

    out = out.drop(columns=empty_cols)

    filename = f"LVC_passive_{name}_v4_1.csv"

    out.to_csv(filename, index=False)

    print(f"Saved {len(out)} rows to {filename}")

Saved 47600 rows to LVC_passive_past_simple_no_adverbs_v4_1.csv
Saved 1523200 rows to LVC_passive_past_simple_with_adverbs_v4_1.csv
Saved 47600 rows to LVC_passive_future_simple_no_adverbs_v4_1.csv
Saved 1523200 rows to LVC_passive_future_simple_with_adverbs_v4_1.csv
Saved 47600 rows to LVC_passive_past_perfect_no_adverbs_v4_1.csv
Saved 1523200 rows to LVC_passive_past_perfect_with_adverbs_v4_1.csv


### active sentences

In [4]:
## this is to build active sentences, with minimal context 

import pandas as pd

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_IN_v1_1.csv"
file2 = "sentence_builders.csv"

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1)
builder = pd.read_csv(file2)

items.columns = items.columns.str.strip()
builder.columns = builder.columns.str.strip()

# Clean FILE1
for col in items.columns:
    if items[col].dtype == "object":
        items[col] = items[col].astype(str).str.strip()

# Clean FILE2
for col in builder.columns:
    if builder[col].dtype == "object":
        builder[col] = builder[col].astype(str).str.strip()

time_specs = (
    builder["time_specs"]
    .replace("nan", pd.NA)
    .dropna()
)
time_specs = time_specs[time_specs != ""].unique().tolist()

subjects = ["they"]

adverbs = (
    builder["adverbs"]
    .replace("nan", pd.NA)
    .dropna()
)
adverbs = adverbs[adverbs != ""].unique().tolist()

# =========================
# VERB FORMS
# =========================

past_forms = {
    "have": "had",
    "receive": "received",
    "take": "took",
    "make": "made",
    "give": "gave"
}

participle_forms = {
    "have": "had",
    "receive": "received",
    "take": "taken",
    "make": "made",
    "give": "given"
}

def get_past_form(verb):
    verb = verb.strip().lower()
    return past_forms.get(verb, verb + "ed")

def get_participle_form(verb):
    verb = verb.strip().lower()
    return participle_forms.get(verb, verb + "ed")

# =========================
# OBJECT PHRASE
# =========================

def build_object_phrase(row):
    obj = row["object"].strip()
    det = str(row["has_determiner"]).strip().lower()

    if det == "no":
        return obj
    else:
        return f"{det} {obj}"

# =========================
# BUILD FILES
# =========================

output_rows = {
    "past_simple_no_adverbs": [],
    "past_simple_with_adverbs": [],
    "future_simple_no_adverbs": [],
    "future_simple_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

for _, item in items.iterrows():

    verb = item["verb"].strip().lower()
    obj = build_object_phrase(item)

    past_verb = get_past_form(verb)
    participle_verb = get_participle_form(verb)

    for time_spec in time_specs:
        for subject in subjects:

            tense_patterns = {
                "past_simple": f"{subject} {{adverb_part}}{past_verb} {obj}",
                "future_simple": f"{subject} will {{adverb_part}}{verb} {obj}",
                "past_perfect": f"{subject} had {{adverb_part}}{participle_verb} {obj}",
            }

            # -------------------------
            # WITHOUT ADVERBS
            # -------------------------
            for tense, pattern in tense_patterns.items():

                sentence_body = pattern.format(adverb_part="")
                sentence = f"{time_spec} {sentence_body}."

                row = item.to_dict()
                row.update({
                    "sentence": sentence,
                    "voice": "active",
                    "tense": tense,
                    "adverb": "NA",
                    "subject": subject,
                    "time_spec": time_spec
                })

                output_rows[f"{tense}_no_adverbs"].append(row)

            # -------------------------
            # WITH ADVERBS
            # -------------------------
            for adverb in adverbs:
                for tense, pattern in tense_patterns.items():

                    sentence_body = pattern.format(adverb_part=f"{adverb} ")
                    sentence = f"{time_spec} {sentence_body}."

                    row = item.to_dict()
                    row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "subject": subject,
                        "time_spec": time_spec
                    })

                    output_rows[f"{tense}_with_adverbs"].append(row)

# =========================
# SAVE FILES
# =========================

for name, rows in output_rows.items():

    out = pd.DataFrame(rows)

    # Remove fully empty columns
    out = out.dropna(axis=1, how="all")

    # Remove columns containing only empty strings
    empty_cols = []

    for col in out.columns:

        values = (
            out[col]
            .astype(str)
            .str.strip()
            .replace("nan", "")
        )

        if (values == "").all():
            empty_cols.append(col)

    out = out.drop(columns=empty_cols)

    filename = f"LVC_active_{name}_v4_1.csv"

    out.to_csv(filename, index=False)

    print(f"Saved {len(out)} rows to {filename}")

Saved 47600 rows to LVC_active_past_simple_no_adverbs_v4_1.csv
Saved 1523200 rows to LVC_active_past_simple_with_adverbs_v4_1.csv
Saved 47600 rows to LVC_active_future_simple_no_adverbs_v4_1.csv
Saved 1523200 rows to LVC_active_future_simple_with_adverbs_v4_1.csv
Saved 47600 rows to LVC_active_past_perfect_no_adverbs_v4_1.csv
Saved 1523200 rows to LVC_active_past_perfect_with_adverbs_v4_1.csv


### active sentences with nominal subjects

In [9]:
import pandas as pd
import random

# =========================
# SETTINGS
# =========================

### for all possible sentences select:
### RANDOM_SUBJECT_SELECTION = False
### RANDOM_OUTPUT_SAMPLE = False

SUBJECT_NUMBER = "both"
# "sg", "pl", "both"

SUBJECT_DET_TYPES = ["det"]#, "det_pl", "det_poss", "no_det"]
# "det", "det_pl", "det_poss", "no_det"

USE_ADVERBS = True

TENSES = ["past_simple"]#, "future_simple", "past_perfect"]

RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True

REMOVE_DUPLICATE_SENTENCES = True

N_RANDOM_ROWS = 1000
RANDOM_STATE = 99

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_IN_v1_1.csv"
file2 = "sentence_builders.csv"

output_file = "LVC_active_subjects_output.csv"

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1)
builder = pd.read_csv(file2)

items.columns = items.columns.str.strip()
builder.columns = builder.columns.str.strip()

for col in items.columns:
    if items[col].dtype == "object":
        items[col] = items[col].astype(str).str.strip()

for col in builder.columns:
    if builder[col].dtype == "object":
        builder[col] = builder[col].astype(str).str.strip()

# =========================
# HELPER
# =========================

def clean_list(series):
    values = (
        series.astype(str)
        .str.strip()
        .replace("nan", pd.NA)
        .dropna()
    )
    values = values[values != ""]
    return values.unique().tolist()

# =========================
# BASIC LISTS
# =========================

time_specs = clean_list(builder["time_specs"])
adverbs = clean_list(builder["adverbs"])

dets = clean_list(builder["det"])
det_pls = clean_list(builder["det_pl"])
det_poss_list = clean_list(builder["det_poss"])

# =========================
# VERB FORMS
# =========================

past_forms = {
    "have": "had",
    "receive": "received",
    "take": "took",
    "make": "made",
    "give": "gave"
}

participle_forms = {
    "have": "had",
    "receive": "received",
    "take": "taken",
    "make": "made",
    "give": "given"
}

def get_past_form(verb):
    return past_forms.get(verb, verb + "ed")

def get_participle_form(verb):
    return participle_forms.get(verb, verb + "ed")

# =========================
# OBJECT PHRASE
# =========================

def build_object_phrase(row):
    obj = row["object"].strip()
    det = str(row["has_determiner"]).strip().lower()

    if det == "no":
        return obj
    else:
        return f"{det} {obj}"

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

for _, row in builder.iterrows():

    subj_sg = str(row.get("subjects_sg", row.get("subject_sg", ""))).strip()
    subj_pl = str(row.get("subjects_pl", row.get("subject_pl", ""))).strip()
    det_poss_tf = str(row.get("det_poss_tf", "")).strip()

    # Singular subjects with generic determiners
    if subj_sg not in ["", "nan"]:
        for det in dets:
            subject_rows.append({
                "subject": f"{det} {subj_sg}",
                "subject_base": subj_sg,
                "subject_number": "sg",
                "subject_det": det,
                "subject_det_type": "det",
                "det_poss_tf": det_poss_tf
            })

    # Singular subjects with possessive determiners
    if subj_sg not in ["", "nan"] and det_poss_tf == "T":
        for det_poss in det_poss_list:
            subject_rows.append({
                "subject": f"{det_poss} {subj_sg}",
                "subject_base": subj_sg,
                "subject_number": "sg",
                "subject_det": det_poss,
                "subject_det_type": "det_poss",
                "det_poss_tf": det_poss_tf
            })

    # Plural subjects without determiner
    if subj_pl not in ["", "nan"]:
        subject_rows.append({
            "subject": subj_pl,
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": "NA",
            "subject_det_type": "no_det",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with generic determiners
    if subj_pl not in ["", "nan"]:
        for det in dets:
            subject_rows.append({
                "subject": f"{det} {subj_pl}",
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": det,
                "subject_det_type": "det",
                "det_poss_tf": det_poss_tf
            })

    # Plural subjects with plural determiners
    if subj_pl not in ["", "nan"]:
        for det_pl in det_pls:
            subject_rows.append({
                "subject": f"{det_pl} {subj_pl}",
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": det_pl,
                "subject_det_type": "det_pl",
                "det_poss_tf": det_poss_tf
            })

    # Plural subjects with possessive determiners
    if subj_pl not in ["", "nan"] and det_poss_tf == "T":
        for det_poss in det_poss_list:
            subject_rows.append({
                "subject": f"{det_poss} {subj_pl}",
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": det_poss,
                "subject_det_type": "det_poss",
                "det_poss_tf": det_poss_tf
            })

subjects_df = pd.DataFrame(subject_rows).drop_duplicates()

# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

subjects_df = subjects_df[
    subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES)
].reset_index(drop=True)

print("Number of available subjects:", len(subjects_df))
print(subjects_df["subject"].head(20))

subject_records = subjects_df.to_dict("records")

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb"].strip().lower()
    obj = build_object_phrase(item)

    past_verb = get_past_form(verb)
    participle_verb = get_participle_form(verb)

    for time_spec in time_specs:

        adverb_options = adverbs if USE_ADVERBS else ["NA"]

        for adverb in adverb_options:
            for tense in TENSES:

                if RANDOM_SUBJECT_SELECTION:
                    subject_iterator = [random.choice(subject_records)]
                else:
                    subject_iterator = subject_records

                for subj_info in subject_iterator:

                    subject = subj_info["subject"]
                    adverb_part = "" if adverb == "NA" else f"{adverb} "

                    if tense == "past_simple":
                        sentence_body = f"{subject} {adverb_part}{past_verb} {obj}"

                    elif tense == "future_simple":
                        sentence_body = f"{subject} will {adverb_part}{verb} {obj}"

                    elif tense == "past_perfect":
                        sentence_body = f"{subject} had {adverb_part}{participle_verb} {obj}"

                    sentence = f"{time_spec} {sentence_body}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "time_spec": time_spec
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

if REMOVE_DUPLICATE_SENTENCES:
    out = out.drop_duplicates(subset=["sentence"])

if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

# Remove empty columns
out = out.dropna(axis=1, how="all")

empty_cols = []

for col in out.columns:
    values = out[col].astype(str).str.strip().replace("nan", "")

    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE
# =========================

out.to_csv(output_file, index=False)

print(f"Saved {len(out)} rows to {output_file}")

Number of available subjects: 200
0          the actor
1         the actors
2        the advisor
3       the advisors
4      the applicant
5     the applicants
6      the architect
7     the architects
8      the assistant
9     the assistants
10       the athlete
11      the athletes
12     the attendant
13    the attendants
14          the aunt
15         the aunts
16         the baker
17        the bakers
18        the barber
19       the barbers
Name: subject, dtype: object
Saved 1000 rows to LVC_active_subjects_output.csv


In [8]:
import pandas as pd
import random

# =========================
# SETTINGS
# =========================

### for all possible sentences select:
### RANDOM_SUBJECT_SELECTION = False
### RANDOM_OUTPUT_SAMPLE = False


SUBJECT_NUMBER = "both"
# options: "sg", "pl", "both"

SUBJECT_DET_TYPES = ["det"]#, "det_pl", "det_poss", "no_det"]
# options: "det", "det_pl", "det_poss", "no_det"

USE_ADVERBS = True
# True = with adverbs
# False = no adverbs only

TENSES = ["past_simple"]#, "future_simple", "past_perfect"]
# options: "past_simple", "future_simple", "past_perfect"

RANDOM_SUBJECT_SELECTION = True
# True  = randomly select one subject for each final sentence row
# False = use all possible subjects

RANDOM_OUTPUT_SAMPLE = True
# True  = save only N_RANDOM_ROWS random rows
# False = save all generated rows

REMOVE_DUPLICATE_SENTENCES = True

N_RANDOM_ROWS = 100
RANDOM_STATE = 99

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_IN_v1_1.csv"
file2 = "sentence_builders.csv"

output_file = "LVC_active_subjects_outputXXX.csv"

# =========================
# SET RANDOM SEED
# =========================

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1)
builder = pd.read_csv(file2)

items.columns = items.columns.str.strip()
builder.columns = builder.columns.str.strip()

for col in items.columns:
    if items[col].dtype == "object":
        items[col] = items[col].astype(str).str.strip()

for col in builder.columns:
    if builder[col].dtype == "object":
        builder[col] = builder[col].astype(str).str.strip()

# =========================
# BASIC LISTS
# =========================

time_specs = (
    builder["time_specs"]
    .replace("nan", pd.NA)
    .dropna()
)
time_specs = time_specs[time_specs != ""].unique().tolist()

adverbs = (
    builder["adverbs"]
    .replace("nan", pd.NA)
    .dropna()
)
adverbs = adverbs[adverbs != ""].unique().tolist()

# =========================
# VERB FORMS
# =========================

past_forms = {
    "have": "had",
    "receive": "received",
    "take": "took",
    "make": "made",
    "give": "gave"
}

participle_forms = {
    "have": "had",
    "receive": "received",
    "take": "taken",
    "make": "made",
    "give": "given"
}

def get_past_form(verb):
    verb = verb.strip().lower()
    return past_forms.get(verb, verb + "ed")

def get_participle_form(verb):
    verb = verb.strip().lower()
    return participle_forms.get(verb, verb + "ed")

# =========================
# OBJECT PHRASE
# =========================

def build_object_phrase(row):
    obj = row["object"].strip()
    det = str(row["has_determiner"]).strip().lower()

    if det == "no":
        return obj
    else:
        return f"{det} {obj}"

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

for _, row in builder.iterrows():

    det = str(row.get("det", "")).strip()
    det_pl = str(row.get("det_pl", "")).strip()
    det_poss_tf = str(row.get("det_poss_tf", "")).strip()
    det_poss = str(row.get("det_poss", "")).strip()

    subj_sg = str(row.get("subjects_sg", "")).strip()
    subj_pl = str(row.get("subjects_pl", "")).strip()

    # Singular subjects with generic determiner
    if subj_sg not in ["", "nan"] and det not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det} {subj_sg}",
            "subject_base": subj_sg,
            "subject_number": "sg",
            "subject_det": det,
            "subject_det_type": "det",
            "det_poss_tf": det_poss_tf
        })

    # Singular subjects with possessive determiner
    if subj_sg not in ["", "nan"] and det_poss_tf == "T" and det_poss not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det_poss} {subj_sg}",
            "subject_base": subj_sg,
            "subject_number": "sg",
            "subject_det": det_poss,
            "subject_det_type": "det_poss",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects without determiner
    if subj_pl not in ["", "nan"]:
        subject_rows.append({
            "subject": subj_pl,
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": "NA",
            "subject_det_type": "no_det",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with generic determiner
    if subj_pl not in ["", "nan"] and det not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det} {subj_pl}",
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": det,
            "subject_det_type": "det",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with plural determiner
    if subj_pl not in ["", "nan"] and det_pl not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det_pl} {subj_pl}",
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": det_pl,
            "subject_det_type": "det_pl",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with possessive determiner
    if subj_pl not in ["", "nan"] and det_poss_tf == "T" and det_poss not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det_poss} {subj_pl}",
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": det_poss,
            "subject_det_type": "det_poss",
            "det_poss_tf": det_poss_tf
        })

subjects_df = pd.DataFrame(subject_rows).drop_duplicates()

# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

subjects_df = subjects_df[
    subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES)
].reset_index(drop=True)

if len(subjects_df) == 0:
    raise ValueError("No subjects left after filtering. Check SUBJECT_NUMBER and SUBJECT_DET_TYPES.")

subject_records = subjects_df.to_dict("records")

def get_subjects_for_row():
    if RANDOM_SUBJECT_SELECTION:
        return [random.choice(subject_records)]
    else:
        return subject_records

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb"].strip().lower()
    obj = build_object_phrase(item)

    past_verb = get_past_form(verb)
    participle_verb = get_participle_form(verb)

    for time_spec in time_specs:

        selected_tenses = [tense for tense in TENSES]

        if USE_ADVERBS:
            adverb_options = adverbs
        else:
            adverb_options = ["NA"]

        for adverb in adverb_options:

            for tense in selected_tenses:

                for subj_info in get_subjects_for_row():

                    subject = subj_info["subject"]

                    if tense == "past_simple":
                        adverb_part = "" if adverb == "NA" else f"{adverb} "
                        sentence_body = f"{subject} {adverb_part}{past_verb} {obj}"

                    elif tense == "future_simple":
                        adverb_part = "" if adverb == "NA" else f"{adverb} "
                        sentence_body = f"{subject} will {adverb_part}{verb} {obj}"

                    elif tense == "past_perfect":
                        adverb_part = "" if adverb == "NA" else f"{adverb} "
                        sentence_body = f"{subject} had {adverb_part}{participle_verb} {obj}"

                    else:
                        raise ValueError(f"Unknown tense: {tense}")

                    sentence = f"{time_spec} {sentence_body}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "time_spec": time_spec,
                        "random_subject_selection": RANDOM_SUBJECT_SELECTION,
                        "random_output_sample": RANDOM_OUTPUT_SAMPLE
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

if REMOVE_DUPLICATE_SENTENCES:
    out = out.drop_duplicates(subset=["sentence"])

if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

# =========================
# REMOVE EMPTY COLUMNS
# =========================

out = out.dropna(axis=1, how="all")

empty_cols = []

for col in out.columns:
    values = (
        out[col]
        .astype(str)
        .str.strip()
        .replace("nan", "")
    )

    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE FILE
# =========================

out.to_csv(output_file, index=False)

print(f"Saved {len(out)} rows to {output_file}")

Saved 100 rows to LVC_active_subjects_outputXXX.csv


In [7]:
import pandas as pd
import random

# =========================
# SETTINGS
# =========================

### for all possible sentences select:
### RANDOM_SUBJECT_SELECTION = False
### RANDOM_OUTPUT_SAMPLE = False


SUBJECT_NUMBER = "sg"
# options: "sg", "pl", "both"

SUBJECT_DET_TYPES = ["det"]#, "det_pl", "det_poss"]#, "no_det"]
# options: "det", "det_pl", "det_poss", "no_det"

USE_ADVERBS = False
# True = with adverbs
# False = no adverbs only

TENSES = ["past_simple"]#, "future_simple", "past_perfect"]
# options: "past_simple", "future_simple", "past_perfect"

RANDOM_SUBJECT_SELECTION = True
# True  = randomly select one subject per item + time_spec
# False = use all possible subjects

RANDOM_OUTPUT_SAMPLE = True
# True  = save only N_RANDOM_ROWS random rows
# False = save all generated rows

N_RANDOM_ROWS = 1000
RANDOM_STATE = 99

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_IN_v1_1.csv"
file2 = "sentence_builders.csv"

output_file = "LVC_active_subjects_output_1000.csv"

# =========================
# SET RANDOM SEED
# =========================

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1)
builder = pd.read_csv(file2)

items.columns = items.columns.str.strip()
builder.columns = builder.columns.str.strip()

for col in items.columns:
    if items[col].dtype == "object":
        items[col] = items[col].astype(str).str.strip()

for col in builder.columns:
    if builder[col].dtype == "object":
        builder[col] = builder[col].astype(str).str.strip()

# =========================
# BASIC LISTS
# =========================

time_specs = (
    builder["time_specs"]
    .replace("nan", pd.NA)
    .dropna()
)
time_specs = time_specs[time_specs != ""].unique().tolist()

adverbs = (
    builder["adverbs"]
    .replace("nan", pd.NA)
    .dropna()
)
adverbs = adverbs[adverbs != ""].unique().tolist()

# =========================
# VERB FORMS
# =========================

past_forms = {
    "have": "had",
    "receive": "received",
    "take": "took",
    "make": "made",
    "give": "gave"
}

participle_forms = {
    "have": "had",
    "receive": "received",
    "take": "taken",
    "make": "made",
    "give": "given"
}

def get_past_form(verb):
    verb = verb.strip().lower()
    return past_forms.get(verb, verb + "ed")

def get_participle_form(verb):
    verb = verb.strip().lower()
    return participle_forms.get(verb, verb + "ed")

# =========================
# OBJECT PHRASE
# =========================

def build_object_phrase(row):
    obj = row["object"].strip()
    det = str(row["has_determiner"]).strip().lower()

    if det == "no":
        return obj
    else:
        return f"{det} {obj}"

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

for _, row in builder.iterrows():

    det = str(row.get("det", "")).strip()
    det_pl = str(row.get("det_pl", "")).strip()
    det_poss_tf = str(row.get("det_poss_tf", "")).strip()
    det_poss = str(row.get("det_poss", "")).strip()

    subj_sg = str(row.get("subjects_sg", "")).strip()
    subj_pl = str(row.get("subjects_pl", "")).strip()

    # Singular subjects with generic determiner
    if subj_sg not in ["", "nan"] and det not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det} {subj_sg}",
            "subject_base": subj_sg,
            "subject_number": "sg",
            "subject_det": det,
            "subject_det_type": "det",
            "det_poss_tf": det_poss_tf
        })

    # Singular subjects with possessive determiner
    if subj_sg not in ["", "nan"] and det_poss_tf == "T" and det_poss not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det_poss} {subj_sg}",
            "subject_base": subj_sg,
            "subject_number": "sg",
            "subject_det": det_poss,
            "subject_det_type": "det_poss",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects without determiner
    if subj_pl not in ["", "nan"]:
        subject_rows.append({
            "subject": subj_pl,
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": "NA",
            "subject_det_type": "no_det",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with generic determiner
    if subj_pl not in ["", "nan"] and det not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det} {subj_pl}",
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": det,
            "subject_det_type": "det",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with plural determiner
    if subj_pl not in ["", "nan"] and det_pl not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det_pl} {subj_pl}",
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": det_pl,
            "subject_det_type": "det_pl",
            "det_poss_tf": det_poss_tf
        })

    # Plural subjects with possessive determiner
    if subj_pl not in ["", "nan"] and det_poss_tf == "T" and det_poss not in ["", "nan"]:
        subject_rows.append({
            "subject": f"{det_poss} {subj_pl}",
            "subject_base": subj_pl,
            "subject_number": "pl",
            "subject_det": det_poss,
            "subject_det_type": "det_poss",
            "det_poss_tf": det_poss_tf
        })

subjects_df = pd.DataFrame(subject_rows).drop_duplicates()

# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

subjects_df = subjects_df[
    subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES)
].reset_index(drop=True)

if len(subjects_df) == 0:
    raise ValueError("No subjects left after filtering. Check SUBJECT_NUMBER and SUBJECT_DET_TYPES.")

subject_records = subjects_df.to_dict("records")

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb"].strip().lower()
    obj = build_object_phrase(item)

    past_verb = get_past_form(verb)
    participle_verb = get_participle_form(verb)

    for time_spec in time_specs:

        if RANDOM_SUBJECT_SELECTION:
            subject_iterator = [random.choice(subject_records)]
        else:
            subject_iterator = subject_records

        for subj_info in subject_iterator:

            subject = subj_info["subject"]

            all_tense_patterns = {
                "past_simple": f"{subject} {{adverb_part}}{past_verb} {obj}",
                "future_simple": f"{subject} will {{adverb_part}}{verb} {obj}",
                "past_perfect": f"{subject} had {{adverb_part}}{participle_verb} {obj}",
            }

            tense_patterns = {
                tense: pattern
                for tense, pattern in all_tense_patterns.items()
                if tense in TENSES
            }

            if USE_ADVERBS:

                for adverb in adverbs:
                    for tense, pattern in tense_patterns.items():

                        sentence = f"{time_spec} {pattern.format(adverb_part=adverb + ' ')}."

                        new_row = item.to_dict()
                        new_row.update(subj_info)
                        new_row.update({
                            "sentence": sentence,
                            "voice": "active",
                            "tense": tense,
                            "adverb": adverb,
                            "time_spec": time_spec,
                            "random_subject_selection": RANDOM_SUBJECT_SELECTION,
                            "random_output_sample": RANDOM_OUTPUT_SAMPLE
                        })

                        rows.append(new_row)

            else:

                for tense, pattern in tense_patterns.items():

                    sentence = f"{time_spec} {pattern.format(adverb_part='')}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": "NA",
                        "time_spec": time_spec,
                        "random_subject_selection": RANDOM_SUBJECT_SELECTION,
                        "random_output_sample": RANDOM_OUTPUT_SAMPLE
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

# REMOVE DUPLICATES FIRST
out = out.drop_duplicates(subset=["sentence"])

# THEN SAMPLE
if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

# =========================
# REMOVE EMPTY COLUMNS
# =========================

out = out.dropna(axis=1, how="all")

empty_cols = []

for col in out.columns:
    values = (
        out[col]
        .astype(str)
        .str.strip()
        .replace("nan", "")
    )

    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE FILE
# =========================

out.to_csv(output_file, index=False)

print(f"Saved {len(out)} rows to {output_file}")

Saved 1000 rows to LVC_active_subjects_output_1000.csv
